Step    Task

1       Load Cleaned Datasets 

2       Skip Existing Date Features

3       Verify Relationship Keys

4       Verify Customer & Product Mapping

5       Merge Datasets (if required)

6       Check Merge Results

7       Prepare Final Analysis Dataset

8       Save Prepared Dataset

In [37]:
import pandas as pd

In [2]:
customers = pd.read_csv("../outputs/cleaned_customers.csv")
products = pd.read_csv("../outputs/cleaned_products.csv")
orders = pd.read_csv("../outputs/cleaned_orders.csv")
order_details = pd.read_csv("../outputs/cleaned_order_details.csv")
targets = pd.read_csv("../outputs/cleaned_targets.csv")

Check Dataset Shape

In [3]:

print("Customers :",customers.shape)
print("Products:",products.shape)
print("Orders:",orders.shape)
print("order_details:",order_details.shape)
print("Targets:",targets.shape)

Customers : (10000, 7)
Products: (1392, 7)
Orders: (400000, 12)
order_details: (1019516, 18)
Targets: (240, 4)


In [40]:
# Orders Table
print("Orders Table Columns:")
print(orders.columns.tolist())

# Order Details Table
print("\nOrder Details Table Columns:")
print(order_details.columns.tolist())

Orders Table Columns:
['Order_ID', 'Order_Date', 'Customer_ID', 'Payment_Method', 'Order_Status', 'Year', 'Month_Name', 'Quarter', 'Day', 'Weekday', 'Month', 'Region']

Order Details Table Columns:
['Order_ID', 'Product_ID', 'Quantity', 'Discount', 'Selling_Price', 'Cost_Price', 'Gross_Sales', 'Discount_Amount', 'Net_Sales', 'Total_Cost', 'Profit', 'Profit_Margin', 'Customer_ID', 'Order_Date', 'Month', 'Month_start', 'Region', 'Category']


Step 2: Prepare Date Columns

Convert Order_Date to Datetime

In [43]:
orders["Order_Date"] = pd.to_datetime(
    orders["Order_Date"],
    errors="coerce"
)
order_details["Order_Date"] = pd.to_datetime(
    order_details["Order_Date"],
    errors="coerce"
)

In [45]:
orders["Order_Date"].head()

0   2023-12-15
1   2023-06-06
2   2023-06-14
3   2023-09-22
4   2025-11-17
Name: Order_Date, dtype: datetime64[us]

Step 3: Verify Relationship Keys

3.1 Check Unique Customer IDs

In [47]:
print("Customers Table:")
print("Total Customer IDs:",customers["Customer_ID"].unique())

print("orders Table:")
print("Total Customer IDs:",orders["Customer_ID"].unique())


Customers Table:
Total Customer IDs: <StringArray>
['CUST000001', 'CUST000002', 'CUST000003', 'CUST000004', 'CUST000005',
 'CUST000006', 'CUST000007', 'CUST000008', 'CUST000009', 'CUST000010',
 ...
 'CUST009991', 'CUST009992', 'CUST009993', 'CUST009994', 'CUST009995',
 'CUST009996', 'CUST009997', 'CUST009998', 'CUST009999', 'CUST010000']
Length: 10000, dtype: str
orders Table:
Total Customer IDs: <StringArray>
['CUST003379', 'CUST008843', 'CUST008484', 'CUST000833', 'CUST009978',
 'CUST006677', 'CUST009188', 'CUST002702', 'CUST004753', 'CUST000736',
 ...
 'CUST007186', 'CUST007771', 'CUST000029', 'CUST003224', 'CUST005818',
 'CUST009053', 'CUST004162', 'CUST004429', 'CUST003130', 'CUST001307']
Length: 9891, dtype: str


3.2 Check Customer IDs Matching

In [51]:
missing_customer_ids = orders[
    ~orders["Customer_ID"].isin(customers["Customer_ID"])
]

print("Unmatched Customer IDs:", len(missing_customer_ids))

Unmatched Customer IDs: 0


3.3 Check Product IDs Matching


In [53]:
missing_product_ids = order_details[
    ~order_details["Product_ID"].isin(products["Product_ID"])
]

print("Unmatched Product IDs:", len(missing_product_ids))

Unmatched Product IDs: 0


3.4 Check Order IDs Matching

In [50]:
missing_order_ids = order_details[
    ~order_details["Order_ID"].isin(orders["Order_ID"])
]

print("Unmatched Order IDs:", len(missing_order_ids))

Unmatched Order IDs: 0


Step 4: Verify Customer & Product Mapping

4.1 Check Customer_ID Consistency

In [6]:
customer_mapping = orders[
    ["Order_ID", "Customer_ID"]
].drop_duplicates()

order_details_customer = order_details[
    ["Order_ID", "Customer_ID"]
].drop_duplicates()

4.2 Merge Both Customer Mappings

In [7]:
customer_check = order_details_customer.merge(
    customer_mapping,
    on="Order_ID",
    how="left",
    suffixes=("_Details", "_Orders")
)

4.3 Find Mismatched Customer IDs

In [8]:
customer_mismatch = customer_check[
    customer_check["Customer_ID_Details"]
    != customer_check["Customer_ID_Orders"]
]

print("Customer ID Mismatches:", len(customer_mismatch))

Customer ID Mismatches: 0


4.4 Display Mismatched Records

In [60]:
customer_mismatch.head()

,Order_ID,Customer_ID_Details,Customer_ID_Orders


Step 5: Merge Datasets

5.1 Merge Order Details with Products

In [9]:
sales_data = order_details.merge(
    products,
    on="Product_ID",
    how="left",
    suffixes=("_Details", "_Product")
)

5.2 Check Merged Dataset

In [11]:
print("Sales Data Shape:", sales_data.shape)

Sales Data Shape: (1019516, 24)


In [12]:
sales_data.head()

,Order_ID,Product_ID,Quantity,Discount,Selling_Price_Details,Cost_Price_Details,Gross_Sales,Discount_Amount,Net_Sales,Total_Cost,...,Month,Month_start,Region,Category_Details,Product_Name,Category_Product,Sub_Category,Brand,Cost_Price_Product,Selling_Price_Product
0,ORD000001,PRD000116,3,0.00,175,135,525,0.0,525.0,405,...,Dec-23,2023-12-01,South,Fruits & Vegetables,Britannia Spinach 1L,Fruits & Vegetables,Vegetables,Britannia,135,175
1,ORD000001,PRD000344,1,0.00,50,45,50,0.0,50.0,45,...,Dec-23,2023-12-01,South,Beverages,Sensodyne Orange Juice 200g,Beverages,Juices,Sensodyne,45,50
2,ORD000001,PRD000756,1,0.05,126,105,126,6.3,119.7,105,...,Dec-23,2023-12-01,South,Staples,Dove Chana Dal Pack of 1,Staples,Pulses,Dove,105,126
3,ORD000002,PRD000878,1,0.15,40,25,40,6.0,34.0,25,...,Jun-23,2023-06-01,West,Bakery,Duracell Chocolate Cookies 1kg,Bakery,Cookies,Duracell,25,40
4,ORD000002,PRD000415,1,0.10,54,35,54,5.4,48.6,35,...,Jun-23,2023-06-01,West,Snacks,Kissan Potato Chips 200g,Snacks,Chips,Kissan,35,54


5.3 Check Product Mapping After Merge

In [13]:
print(
    "Missing Product Names:",
    sales_data["Product_Name"].isna().sum()
)

Missing Product Names: 0


5.4 Check Columns

In [14]:
sales_data.columns.tolist()

['Order_ID',
 'Product_ID',
 'Quantity',
 'Discount',
 'Selling_Price_Details',
 'Cost_Price_Details',
 'Gross_Sales',
 'Discount_Amount',
 'Net_Sales',
 'Total_Cost',
 'Profit',
 'Profit_Margin',
 'Customer_ID',
 'Order_Date',
 'Month',
 'Month_start',
 'Region',
 'Category_Details',
 'Product_Name',
 'Category_Product',
 'Sub_Category',
 'Brand',
 'Cost_Price_Product',
 'Selling_Price_Product']

Step 6: Merge Sales Data with Orders

In [16]:
sales_data.head()

,Order_ID,Product_ID,Quantity,Discount,Selling_Price_Details,Cost_Price_Details,Gross_Sales,Discount_Amount,Net_Sales,Total_Cost,...,Month,Month_start,Region,Category_Details,Product_Name,Category_Product,Sub_Category,Brand,Cost_Price_Product,Selling_Price_Product
0,ORD000001,PRD000116,3,0.00,175,135,525,0.0,525.0,405,...,Dec-23,2023-12-01,South,Fruits & Vegetables,Britannia Spinach 1L,Fruits & Vegetables,Vegetables,Britannia,135,175
1,ORD000001,PRD000344,1,0.00,50,45,50,0.0,50.0,45,...,Dec-23,2023-12-01,South,Beverages,Sensodyne Orange Juice 200g,Beverages,Juices,Sensodyne,45,50
2,ORD000001,PRD000756,1,0.05,126,105,126,6.3,119.7,105,...,Dec-23,2023-12-01,South,Staples,Dove Chana Dal Pack of 1,Staples,Pulses,Dove,105,126
3,ORD000002,PRD000878,1,0.15,40,25,40,6.0,34.0,25,...,Jun-23,2023-06-01,West,Bakery,Duracell Chocolate Cookies 1kg,Bakery,Cookies,Duracell,25,40
4,ORD000002,PRD000415,1,0.10,54,35,54,5.4,48.6,35,...,Jun-23,2023-06-01,West,Snacks,Kissan Potato Chips 200g,Snacks,Chips,Kissan,35,54


Step 6.1 — Select Required Columns from Orders

In [17]:
orders_selected = orders[
    [
        "Order_ID",
        "Payment_Method",
        "Order_Status"
    ]
]

Step 6.2 — Check Duplicate Order IDs

In [18]:
orders_selected["Order_ID"].duplicated().sum()

np.int64(0)

Step 6.3 — Merge Sales Data with Orders

In [20]:
sales_data = sales_data.merge(
    orders_selected,
    on = "Order_ID",
    how = "left",
    validate ="many_to_one"
)

Step 6.4: Verify Merged Columns

6.4.1 Check New Columns

In [21]:
sales_data.columns.tolist()

['Order_ID',
 'Product_ID',
 'Quantity',
 'Discount',
 'Selling_Price_Details',
 'Cost_Price_Details',
 'Gross_Sales',
 'Discount_Amount',
 'Net_Sales',
 'Total_Cost',
 'Profit',
 'Profit_Margin',
 'Customer_ID',
 'Order_Date',
 'Month',
 'Month_start',
 'Region',
 'Category_Details',
 'Product_Name',
 'Category_Product',
 'Sub_Category',
 'Brand',
 'Cost_Price_Product',
 'Selling_Price_Product',
 'Payment_Method',
 'Order_Status']

6.4.2 Check Missing Values

In [22]:
print(
    "Missing Payment Method:",
    sales_data["Payment_Method"].isna().sum()
)
print(
    "Missing Order Status:",
    sales_data["Order_Status"].isna().sum()
)

Missing Payment Method: 0
Missing Order Status: 0



6.4.3 Compare Row Count

In [23]:
print("Rows after merge:", len(sales_data))
print("Rows before merge:", len(order_details))

Rows after merge: 1019516
Rows before merge: 1019516


Step 7: Final Sales Dataset Verification

7.1 Check Required Columns

In [24]:
required_columns = [
    "Order_ID",
    "Product_ID",
    "Customer_ID",
    "Product_Name",
    "Payment_Method",
    "Order_Status",
    "Quantity",
    "Net_Sales",
    "Profit"
]

for col in required_columns:
    print(col, ":", col in sales_data.columns)

Order_ID : True
Product_ID : True
Customer_ID : True
Product_Name : True
Payment_Method : True
Order_Status : True
Quantity : True
Net_Sales : True
Profit : True


7.2 Check Missing Values

In [25]:
sales_data[
    ["Order_ID", "Product_ID", "Customer_ID",
     "Product_Name", "Payment_Method", "Order_Status"]
].isna().sum()

Order_ID          0
Product_ID        0
Customer_ID       0
Product_Name      0
Payment_Method    0
Order_Status      0
dtype: int64

7.3 Check Duplicate Rows

In [26]:
print("Duplicate Rows:", sales_data.duplicated().sum())

Duplicate Rows: 0


7.4 Display Final Dataset

In [27]:
sales_data.head()

,Order_ID,Product_ID,Quantity,Discount,Selling_Price_Details,Cost_Price_Details,Gross_Sales,Discount_Amount,Net_Sales,Total_Cost,...,Region,Category_Details,Product_Name,Category_Product,Sub_Category,Brand,Cost_Price_Product,Selling_Price_Product,Payment_Method,Order_Status
0,ORD000001,PRD000116,3,0.00,175,135,525,0.0,525.0,405,...,South,Fruits & Vegetables,Britannia Spinach 1L,Fruits & Vegetables,Vegetables,Britannia,135,175,Credit Card,Delivered
1,ORD000001,PRD000344,1,0.00,50,45,50,0.0,50.0,45,...,South,Beverages,Sensodyne Orange Juice 200g,Beverages,Juices,Sensodyne,45,50,Credit Card,Delivered
2,ORD000001,PRD000756,1,0.05,126,105,126,6.3,119.7,105,...,South,Staples,Dove Chana Dal Pack of 1,Staples,Pulses,Dove,105,126,Credit Card,Delivered
3,ORD000002,PRD000878,1,0.15,40,25,40,6.0,34.0,25,...,West,Bakery,Duracell Chocolate Cookies 1kg,Bakery,Cookies,Duracell,25,40,UPI,Delivered
4,ORD000002,PRD000415,1,0.10,54,35,54,5.4,48.6,35,...,West,Snacks,Kissan Potato Chips 200g,Snacks,Chips,Kissan,35,54,UPI,Delivered


In [28]:
print("Final Dataset Shape:", sales_data.shape)

Final Dataset Shape: (1019516, 26)


8.1 Check Available Columns

In [29]:
sales_data.columns.tolist()

['Order_ID',
 'Product_ID',
 'Quantity',
 'Discount',
 'Selling_Price_Details',
 'Cost_Price_Details',
 'Gross_Sales',
 'Discount_Amount',
 'Net_Sales',
 'Total_Cost',
 'Profit',
 'Profit_Margin',
 'Customer_ID',
 'Order_Date',
 'Month',
 'Month_start',
 'Region',
 'Category_Details',
 'Product_Name',
 'Category_Product',
 'Sub_Category',
 'Brand',
 'Cost_Price_Product',
 'Selling_Price_Product',
 'Payment_Method',
 'Order_Status']

8.2 Select Important Columns

In [30]:
analysis_columns = [
    "Order_ID",
    "Product_ID",
    "Product_Name",
    "Customer_ID",
    "Order_Date",
    "Payment_Method",
    "Order_Status",
    "Quantity",
    "Discount",
    "Selling_Price_Details",
    "Cost_Price_Details",
    "Gross_Sales",
    "Discount_Amount",
    "Net_Sales",
    "Total_Cost",
    "Profit",
    "Profit_Margin",
    "Region",
    "Category_Details",
    "Sub_Category",
    "Brand"
]

8.3 Create Final Analysis Dataset

In [31]:
final_sales_data = sales_data[analysis_columns].copy()

In [32]:
final_sales_data.head()

,Order_ID,Product_ID,Product_Name,Customer_ID,Order_Date,Payment_Method,Order_Status,Quantity,Discount,Selling_Price_Details,...,Gross_Sales,Discount_Amount,Net_Sales,Total_Cost,Profit,Profit_Margin,Region,Category_Details,Sub_Category,Brand
0,ORD000001,PRD000116,Britannia Spinach 1L,CUST003379,2023-12-15,Credit Card,Delivered,3,0.00,175,...,525,0.0,525.0,405,120.0,0.228571,South,Fruits & Vegetables,Vegetables,Britannia
1,ORD000001,PRD000344,Sensodyne Orange Juice 200g,CUST003379,2023-12-15,Credit Card,Delivered,1,0.00,50,...,50,0.0,50.0,45,5.0,0.100000,South,Beverages,Juices,Sensodyne
2,ORD000001,PRD000756,Dove Chana Dal Pack of 1,CUST003379,2023-12-15,Credit Card,Delivered,1,0.05,126,...,126,6.3,119.7,105,14.7,0.122807,South,Staples,Pulses,Dove
3,ORD000002,PRD000878,Duracell Chocolate Cookies 1kg,CUST008843,2023-06-06,UPI,Delivered,1,0.15,40,...,40,6.0,34.0,25,9.0,0.264706,West,Bakery,Cookies,Duracell
4,ORD000002,PRD000415,Kissan Potato Chips 200g,CUST008843,2023-06-06,UPI,Delivered,1,0.10,54,...,54,5.4,48.6,35,13.6,0.279835,West,Snacks,Chips,Kissan


In [33]:
print("Final Analysis Dataset Shape:", final_sales_data.shape)

Final Analysis Dataset Shape: (1019516, 21)


Step 9: Save Prepared Dataset

In [34]:
final_sales_data.to_csv(
    "../outputs/prepared_sales_data.csv",
    index= False
)

In [35]:
print("Rows:", final_sales_data.shape[0])
print("Columns:", final_sales_data.shape[1])
print("Missing Values:", final_sales_data.isna().sum().sum())
print("Duplicate Rows:", final_sales_data.duplicated().sum())

Rows: 1019516
Columns: 21
Missing Values: 0
Duplicate Rows: 0


Prepared Sales Dataset saved successfully!
